In [1]:
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()
import sys
sys.path.append("../")
from dotenv import load_dotenv
load_dotenv()

from src.llm_utils import LLM_wrapper, RepoAssessment
from src.llm_utils import REPO_ASSESSMENT_PROMPT as repo_assessment_prompt
from src.repo_utils import RepoStatus

In [2]:
MODEL_NAME = "gpt-5.2-2025-12-11"
CONTEXT_WINDOW = 395000 # gpt 5.2's context window is 400k tokens

In [3]:
llm_wrapper = LLM_wrapper(
    model_name=MODEL_NAME,
    system_prompt=repo_assessment_prompt,
    output_format_class=RepoAssessment
)

In [4]:
df_pred = llm_wrapper.assess_dataframe(
    input_file_path="../data/03_repo_assessment/gt_repo_assessment.csv",
    text_column="repo_content",
    output_dir="../data/03_repo_assessment/",
    # we only want to process rows that were properly cloned
    row_filter=lambda row: row["repo_status"] == RepoStatus.OK
)

Total rows: 35, Already processed: 0, Remaining (after filter): 29


100%|██████████| 29/29 [03:33<00:00,  7.35s/it]

Processing complete. Final results saved.


In [ ]:
# Similar to 2b, include code to read df_pred directly. 

In [3]:
df_pred

NameError: name 'df_pred' is not defined

# Not downloaded by utility

In [4]:
df_pred_notok = df_pred[df_pred["repo_status"] != RepoStatus.OK]

NameError: name 'df_pred' is not defined

In [7]:
# Should have been captured, but was missed by repo util
df_pred_notok[df_pred_notok["gt_is_empty"] == False][["repo_url", "repo_status", "repo_error"]]

,repo_url,repo_status,repo_error
13,https://doi.org/10.57770/QTYQZS,inaccessible,HTTPSConnectionPool(host='dataverse.lib.nycu.e...


In [8]:
# These are not accessible by humans / not code - it is normal that the repo util skipped them
df_pred_notok[df_pred_notok["gt_is_empty"] == True]

,PMCID,repo_url,gt_is_empty,gt_contains_readme,gt_readme_purpose_and_outputs,gt_contains_requirements,gt_requirements_dependency_versions,gt_contains_license,gt_sufficient_code_documentation,gt_is_modular_and_structured,...,sufficient_code_documentation,is_modular_and_structured,implements_tests,fixes_seed_if_stochastic,lists_hardware_requirements,contains_link_to_paper,contains_citation,includes_data_or_sample,comments_and_explanations,coding_languages
0,PMC11307559,https://seakheeoh76.shinyapps.io/XGBoost_BA_pr...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,PMC8175333,https://github.com/ATARIQ5,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
19,PMC11992218,https://www.cdc.gov/nchs/nhanes/index.htm,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
21,PMC10844815,https://github.com/yiwangz/SS_learning/MLinHCT,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
30,PMC8784888,https://github.com/AkihiroShiroshita/Predictio...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


# Empty

In [20]:
df_pred[df_pred["gt_is_empty"] == True][["gt_is_empty", "is_empty"]]

,gt_is_empty,is_empty
0,True,<NA>
2,True,True
3,True,<NA>
19,True,<NA>
21,True,<NA>
30,True,<NA>


# Performance analysis

In [9]:
df_pred_ok = df_pred[df_pred["repo_status"] == RepoStatus.OK]

In [22]:
col = "contains_link_to_paper"
df_pred_ok[["repo_url", col, "gt_" + col]][df_pred_ok["gt_" + col] != df_pred_ok[col]]

,repo_url,contains_link_to_paper,gt_contains_link_to_paper
7,https://github.com/IreneSophia/MLSPE,True,False
10,https://doi.org/10.5281/zenodo.4560078,True,False
14,https://osf.io/w7aes/,True,False
16,https://github.com/putzfn/HNLNL_autosegmentati...,True,False
27,https://github.com/tsryo/evalFL,True,False
29,https://github.com/weissman-lab/restriction,True,False


In [18]:
import numpy as np
import pandas as pd

fields = [
    "contains_readme",
    "readme_purpose_and_outputs",
    "contains_requirements",
    "requirements_dependency_versions",
    "contains_license",
    "sufficient_code_documentation",
    "is_modular_and_structured",
    "implements_tests",
    "fixes_seed_if_stochastic",
    "lists_hardware_requirements",
    "contains_link_to_paper",
    "contains_citation",
    "includes_data_or_sample",
]

metrics = []

for field in fields:
    gt_col = f"gt_{field}"
    pred_col = f"{field}"

    # Drop rows where either GT or prediction is NaN for this field
    sub = df_pred_ok[[gt_col, pred_col]].dropna()

    if sub.empty:
        continue

    y_true = sub[gt_col].astype(bool)
    y_pred = sub[pred_col].astype(bool)

    tp = ((y_true) & (y_pred)).sum()
    tn = ((~y_true) & (~y_pred)).sum()
    fp = ((~y_true) & (y_pred)).sum()
    fn = ((y_true) & (~y_pred)).sum()

    accuracy = (y_true == y_pred).mean()
    recall_true = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    precision_true = tp / (tp + fp) if (tp + fp) > 0 else np.nan

    f1 = 2 * (precision_true * recall_true) / (precision_true + recall_true) if (precision_true + recall_true) > 0 else np.nan
    
    metrics.append(
        {
            "field": field,
            "accuracy": accuracy,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "recall_true": recall_true,
            "precision_true": precision_true,
            "f1": f1,
            "n_non_null": len(sub),

        }
    )
metrics_df = pd.DataFrame(metrics).set_index("field")
display(metrics_df.round(3))
# Calculate macro-averaged F1 score
macro_f1 = metrics_df['f1'].mean()
print(f"\nMacro-averaged F1 score: {macro_f1:.3f}")

# Calculate weighted F1 score
weighted_f1 = (metrics_df['f1'] * metrics_df['n_non_null']).sum() / metrics_df['n_non_null'].sum()
print(f"Weighted F1 score: {weighted_f1:.3f}")

# Calculate weighted accuracy, recall, and precision
weighted_accuracy = (metrics_df['accuracy'] * metrics_df['n_non_null']).sum() / metrics_df['n_non_null'].sum()
weighted_recall = (metrics_df['recall_true'] * metrics_df['n_non_null']).sum() / metrics_df['n_non_null'].sum()
weighted_precision = (metrics_df['precision_true'] * metrics_df['n_non_null']).sum() / metrics_df['n_non_null'].sum()

print(f"Weighted Accuracy: {weighted_accuracy:.3f}")
print(f"Weighted Recall: {weighted_recall:.3f}")
print(f"Weighted Precision: {weighted_precision:.3f}")


,accuracy,tp,tn,fp,fn,recall_true,precision_true,f1,n_non_null
field,,,,,,,,,
contains_readme,1.000,24,5,0,0,1.000,1.000,1.000,29
readme_purpose_and_outputs,0.750,14,4,6,0,1.000,0.700,0.824,24
contains_requirements,0.828,8,16,5,0,1.000,0.615,0.762,29
requirements_dependency_versions,0.875,6,1,0,1,0.857,1.000,0.923,8
contains_license,1.000,8,21,0,0,1.000,1.000,1.000,29
sufficient_code_documentation,0.724,10,11,4,4,0.714,0.714,0.714,29
is_modular_and_structured,0.897,17,9,0,3,0.850,1.000,0.919,29
implements_tests,1.000,1,28,0,0,1.000,1.000,1.000,29
fixes_seed_if_stochastic,0.727,10,6,4,2,0.833,0.714,0.769,22



Macro-averaged F1 score: 0.831
Weighted F1 score: 0.827
Weighted Accuracy: 0.881
Weighted Recall: 0.913
Weighted Precision: 0.796


In [12]:
df_pred.to_csv("../data/03_repo_assessment/evaluation_repo_assessment.csv", index=False)